## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent

agent = create_agent(
    model='ollama:llama3.1:8b',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

In the world of the Galactic Union of Advanced Lifeforms (GUAL), the capital of the Moon, now officially known as Lunar Terra, is a majestic city called Selenea. Located in the vast, cratered expanse of the Sea of Tranquility, Selenea is a marvel of engineering and architecture.

**Geography and Climate**
Selenea is nestled between the Apennine Mountains and the Tycho Crater, with the city's lower levels extending into the lunar regolith. The city's terrain is divided into five distinct districts, each with its unique features and climate. The central district, the heart of the city, is a massive, dome-covered metropolis with a self-sustaining atmosphere. The other districts are:

1. **Nova Terra**: A sprawling, elevated neighborhood that offers breathtaking views of the lunar horizon.
2. **Lunar Springs**: A lush, verdant oasis that provides a tranquil retreat from the hustle and bustle of city life.
3. **Astrum**: A vibrant, futuristic district filled with cutting-edge research facil

## Image input

In [5]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [6]:
print(uploader.value)

({'name': 'lunar-10056714_1280.jpg', 'type': 'image/jpeg', 'size': 433052, 'content': <memory at 0x000001F6A7C33340>, 'last_modified': datetime.datetime(2026, 8, 28, 20, 57, 38, 157000, tzinfo=datetime.timezone.utc)},)


In [7]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [10]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/jpg"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

ResponseError: {"error":{"code":400,"message":"Multimodal data provided, but model does not support multimodal requests.","type":"invalid_request_error"}} (status code: 400)

## Audio input

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

In [ ]:
agent = create_agent(
    model='gpt-audio',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)